In [1]:
import pandas as pd
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
import numpy as np
rc = ReportClass()


In [2]:
ruta = rc.validar_ruta()
    

In [3]:
ventas = pd.read_excel(ruta/'file'/'BASE VENTAS 2025.xlsx')

In [4]:
conta = pd.read_csv(ruta/'data'/'contabilidad'/'base_consolidada.csv', sep=';', decimal=',', encoding='utf-8')

C:\Users\Dataa\AppData\Local\Temp\ipykernel_26456\3996170896.py:1: DtypeWarning: Columns (4,5,11,13,21,24,26,28,29,30,31,32,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  conta = pd.read_csv(ruta/'data'/'contabilidad'/'base_consolidada.csv', sep=';', decimal=',', encoding='utf-8')


In [5]:
conta['Balance'] = conta['Balance'].fillna(0)

In [6]:
conta['Balance'] = conta['Balance'].astype('float')

In [7]:
conta.loc[conta['Concepto']=='Ingreso operativo', 'Concepto'] = 'Ingreso Operativo'

In [8]:
conta['Concepto'] = conta['Concepto'].str.strip()

In [9]:
ventas.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
       'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
       'ZONA', 'ASESOR COMERCIAL'],
      dtype='object')

In [10]:
# Agregar el costo por producto a la base de ventas

In [11]:
kits = pd.read_excel(ruta/'data'/'kits.xlsx')

In [12]:
costos = pd.read_excel(ruta/'data'/'costos_productos.xlsx')

In [13]:
# Agrega los costos por producto a la base de kits
kits['PRODUCTO_SIN_SKU'] =kits['PRODUCTO'].apply(lambda x: re.search(r'\]\s*(.*)', x).group(1) if re.search(r'\]\s*(.*)', x) else '').str.strip()
kits = kits.merge(costos[['Nombre', 'Costo']], left_on='PRODUCTO_SIN_SKU', right_on='Nombre').drop(columns=['Nombre'])
costos_kit = kits.groupby('KIT').sum().reset_index()[['KIT', 'Costo']]

In [14]:
# Agrega los costos por producto a la base de productos
productos = pd.read_excel(ruta/'data'/'cuentas_clave'/'base_cuentas_clave.xlsx') 
productos['PRODUCTO_SIN_SKU'] =productos['PRODUCTO'].apply(lambda x: re.search(r'\]\s*(.*)', x).group(1) if re.search(r'\]\s*(.*)', x) else '').str.strip()
productos = productos.merge(costos[['Nombre', 'Costo']], left_on='PRODUCTO_SIN_SKU', right_on='Nombre', how='left').drop(columns=['Nombre'])
productos = productos.merge(costos_kit, left_on='PRODUCTO', right_on='KIT', how='left').drop(columns=['KIT'])
productos['costos'] = np.where(productos['Costo_x']==0, productos['Costo_y'], productos['Costo_x'])
# Guardar el archivo con los costos agregados
productos[['PRODUCTO', 'Costo_x','Costo_y','costos']].to_excel(ruta/'file'/'productos_con_costos.xlsx', index=False)

In [15]:
# agregar los costos a la base de ventas
ventas = ventas.merge(productos[['PRODUCTO', 'costos']], on='PRODUCTO', how='left')

In [16]:
ventas['COSTO_TOTAL'] = ventas['costos'] * ventas['CANTIDAD']

In [17]:
ventas['CLIENTE'] = ventas['CLIENTE'].str.upper().str.strip().str.replace(r'\s+',' ', regex=True)   

In [18]:
ventas_agru =ventas.groupby('NUMERO_FACTURA').agg({
    'IDENTIFICACION_CLIENTE':'first',
    'CLIENTE':'first',
    'CATEGORÍA':'first',
    'costos':'sum',
    'TOTAL($)':'sum'}).reset_index()

In [19]:
ventas.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
       'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
       'ZONA', 'ASESOR COMERCIAL', 'costos', 'COSTO_TOTAL'],
      dtype='object')

In [20]:
clientes_categoria = ventas.groupby('CLIENTE').agg({'CATEGORÍA':'first', }).reset_index()

In [21]:
# base_categoria_conta = conta.merge(clientes_categoria, left_on='Número',right_on='IDENTIFICACION_CLIENTE', how='left')

In [22]:
# base_merge = conta.merge(ventas_agru, how='left', left_on='Número', right_on='NUMERO_FACTURA')

In [23]:
conta['CLIENTE'] = conta['Contacto'].str.upper().str.strip().str.replace(r'\s+',' ', regex=True)

In [24]:
conta['Balance'].sum()

np.float64(-22032384951.199993)

In [45]:
base_categoria_conta = conta.merge(clientes_categoria, on='CLIENTE', how='left')

In [47]:
base_categoria_conta.columns

Index(['Fecha', 'Diario', 'Número', 'Cuenta Origen', 'Creado el', 'NIT',
       'Contacto', 'Referencia', 'Etiqueta', 'Importe en divisa', 'Débito',
       'Crédito', 'Balance', 'Distribución analítica ori', 'Creado por',
       'Cuenta', 'Nombre cuenta', 'N1', 'N2', 'N3', 'Nivel', 'Niveles',
       'Distribución analítica', 'Nombre Cencosto', 'ADM/VTAS', 'Origen',
       'Nombre cuenta concepto', 'Concepto', 'CC', 'Unnamed: 4', 'tipo',
       'Detalle', 'Mes', 'Año', 'Unnamed: 27', 'Unnamed: 28', 'CLIENTE',
       'CATEGORÍA'],
      dtype='object')

In [27]:
# base_categoria_conta = base_categoria_conta.groupby(['Concepto', 'CLIENTE']).agg({'Contacto':'first','CATEGORÍA':'first','Balance':'sum'}).reset_index()

In [28]:
# batase_categoria_con = base_categoria_conta[~base_categoria_conta['Concepto'].isin(['Costo directo de ventas', 'Ingreso Operativo'])]

In [48]:
base_categoria_conta_agru = base_categoria_conta.groupby(['Concepto','CLIENTE'], dropna=False).agg({'Origen':'first','Contacto':'first','CATEGORÍA':'first','Balance':'sum'}).reset_index()

In [49]:
base_categoria_conta_agru['Balance'] *= -1

In [50]:
base_categoria_conta_agru.to_excel(r"C:\Users\Dataa\Desktop\PYG2.xlsx", index=False)